<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/03_limpieza/ENARES_2024_CRS04_STAGE03_NB02_STAGE03_ANALYTICAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ============================================================
# ENARES 2024 - CRS04
# STAGE 03 - NB02VALIDACION Y SETUP
# ============================================================

from google.colab import auth
from google.cloud import bigquery
from datetime import datetime, timezone
import pandas as pd
import os

auth.authenticate_user()

PROJECT_ID = os.environ.get("PROJECT_ID") or input("Enter your Google Cloud PROJECT_ID: ").strip()
if not PROJECT_ID:
    raise ValueError("PROJECT_ID cannot be empty.")
LOCATION = "US"

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

RUN_UTC = datetime.now(timezone.utc).isoformat()

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"

LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs/stage03"
SQL_DIR = f"{ROOT_DRIVE}/02SQL"
R_DIR = f"{ROOT_DRIVE}/03Scripts_R"
OUTPUT_DIR = f"{ROOT_DRIVE}/04Outputs"

for folder in [LOG_DIR, SQL_DIR, R_DIR, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("PROJECT:", PROJECT_ID)
print("RUN UTC:", RUN_UTC)

Enter your Google Cloud PROJECT_ID: enares-2024-crs04
PROJECT: enares-2024-crs04
RUN UTC: 2026-06-21T02:37:58.842187+00:00


In [11]:
# ============================================================
# ISSUE #30
# VERIFICAR TABLA CLEANED
# ============================================================

client.query(f"""
SELECT
    COUNT(*) AS rows_cleaned
FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`
""").result().to_dataframe()

,rows_cleaned
0,18807


In [12]:
# ============================================================
# ISSUE #30
# CREAR ANALYTICAL VERSION 1
# ============================================================

analytical_sql = f"""
CREATE OR REPLACE TABLE
`{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents`
AS

WITH base AS (

SELECT *
FROM `{PROJECT_ID}.enares2024_crs04_cleaned.cleaned_crs04_merged_adolescents`

),

recodes AS (

SELECT
*,

CASE
    WHEN C3P301_4 = 1 THEN 1
    WHEN C3P301_4 = 2 THEN 0
    WHEN C3P301_4 = 3 THEN NULL
END AS justifica_castigo_docente,

CASE
    WHEN C3P301_5 = 1 THEN 1
    WHEN C3P301_5 = 2 THEN 0
    WHEN C3P301_5 = 3 THEN NULL
END AS justifica_castigo_parental

FROM base

),

indicadores AS (

SELECT
*,

CASE
    WHEN justifica_castigo_docente IS NULL
     AND justifica_castigo_parental IS NULL
    THEN NULL

    WHEN COALESCE(justifica_castigo_docente,0)=1
      OR COALESCE(justifica_castigo_parental,0)=1
    THEN 1

    ELSE 0
END AS justifica_al_menos_una

FROM recodes

)

SELECT *
FROM indicadores
"""

In [13]:
client.query(analytical_sql).result()

print("analytical creada")

analytical creada


In [14]:
client.query(f"""
SELECT
COUNT(*) AS rows_analytical
FROM `{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents`
""").result().to_dataframe()

,rows_analytical
0,18807


In [15]:
resultado = client.query(f"""
SELECT
    justifica_al_menos_una,
    COUNT(*) AS n
FROM `{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents`
GROUP BY 1
ORDER BY 1
""").result().to_dataframe()

display(resultado)

,justifica_al_menos_una,n
0,<NA>,34
1,0,10033
2,1,8740
